## Batch scripts: Utilities and Packages (LightGBM benchmark, Olink -> SomaScan)

In [ ]:
# parameters
batch_number = 0

In [ ]:
import os
import sys
## adding the root directory to the library directory so still can read utility functions
sys.path.append(os.path.abspath(os.path.join(os.path.dirname("..."), "..")))
project_root = os.path.abspath(os.path.expanduser("..."))
sys.path.append(project_root)
os.chdir(project_root) # setting work directory

import warnings
import pickle
import pandas as pd
import numpy as np
import lightgbm as lgb

def fxn():
    warnings.warn("deprecated", DeprecationWarning)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    fxn()

In [ ]:
# Load Olink and Soma assays from the MESA cohort

MESA_Olink  = pd.read_csv("Data/final_datasets/MESA_Olink_3k_log2_nodup_rm10na_knn_final.tsv.gz", sep = "\t",
                          compression = "gzip")
MESA_Soma = pd.read_csv("Data/final_datasets/MESA_SOMA_5k_log2scaled_nodup_final.tsv.gz", sep = "\t",
                        compression = "gzip").drop(columns={"bmi"}, axis=1)

In [ ]:
# Find subjects who are in both datasets
overlap_IDs = np.unique(list(set(np.array(MESA_Olink["id"])).intersection(set(np.array(MESA_Soma["id"])))))

In [ ]:
# Drop repeated subject IDs in both cohorts, confirm alignment

MESA_Olink  = MESA_Olink[MESA_Olink["id"].isin(overlap_IDs)].sort_values(by=["id"]).drop_duplicates(["id"])
MESA_Soma   = MESA_Soma[MESA_Soma["id"].isin(overlap_IDs)].sort_values(by=["id"]).drop_duplicates(["id"])

print(MESA_Olink.shape)
print(MESA_Soma.shape)

ids1 = list(MESA_Olink["id"])
ids2 = list(MESA_Soma["id"])

if ids1 == ids2:
    print("ids in the same order")
else:
    print("ids not in the same order")

In [ ]:
MESA_Soma = MESA_Soma.set_index("id")
MESA_Olink = MESA_Olink.set_index("id")

if MESA_Soma.index.equals(MESA_Olink.index):
    print("yes")
else:
    print("no")

In [ ]:
## loading in training and testing index -- SAME pickled splits used for HGB, so results are directly comparable
train_index = pickle.load(open(f"Data/helper_data/train_index.pkl", "rb"))
test_index = pickle.load(open(f"Data/helper_data/test_index.pkl", "rb"))

print(train_index[1:20])

In [ ]:
## loading prot list -- SAME chunking scheme as HGB batches
protlist = pickle.load(open(f"Data/helper_data/os_protlist_chunk{batch_number}.pkl", "rb"))
print(protlist[0:10])
print(len(protlist))

## Training loop (LightGBM, Olink -> SomaScan)

In [ ]:
## defining X
X = MESA_Olink
print(MESA_Olink.shape)

In [ ]:
## define function to train LightGBM
## NOTE: default hyperparameters (no tuning), matching the same "reasonable defaults" framing
## used for HGB, so this is an apples-to-apples benchmark rather than a tuned-vs-untuned comparison
def train_olink_to_soma_lgb(protein_name):
    y = MESA_Soma[protein_name]
    if y.loc[train_index].isna().any():
        print(f"skip protein {protein_name} due to NA in SOMA data")
        return

    X_train = X.loc[train_index]
    y_train = y.loc[train_index]

    ## LightGBM regressor, default-ish settings comparable to sklearn's HGB defaults
    reg = lgb.LGBMRegressor(random_state=0, verbosity=-1)
    reg.fit(X_train, y_train)

    model_path = f"Data/soma_trained_models/olink_soma/3kolink_5k_log2scaled_soma_lgb/{protein_name}_lgb.pkl"
    os.makedirs(os.path.dirname(model_path), exist_ok=True)
    pickle.dump(reg, open(model_path, "wb"))
    print(f"Saved model for {protein_name} to {model_path}")

In [ ]:
%%time

for p in protlist:
    train_olink_to_soma_lgb(p)